# 여러 종류 학습 (Colab, L4)

zip 에 든 종류마다 차례로: 공개된 Laya 체크포인트 두 개(`base`, `typed`) 중 validation 제로샷 정확도가 높은 쪽에서 파인튜닝하고, 학습 전후 validation 을 채점합니다. calibration split 이 있으면 그것으로 온도를 맞춰 확신도를 보정합니다. 유료 호출은 없고, 게이트 테스트는 읽지 않습니다.

## 준비 (PC)

    python laya/train.py pack analysis ask conflict progress redirect route source useful coverage

`laya/data/colab.zip` 을 Google Drive 의 `내 드라이브/laya/` 에 올립니다. 종류는 위 명령에 적은 순서대로 학습합니다. `coverage` 는 상태가 길어(8,192 토큰) 가장 오래 걸리므로 마지막에 둡니다.

## 실행

1. 런타임 → 런타임 유형 변경 → L4 GPU
2. 런타임 → 모두 실행 (Drive 연결 권한은 허용)

같은 런타임에서 다시 돌려도 됩니다. 실행마다 체크포인트를 `/content/laya/checkpoints/<시각>-…/` 에 따로 두므로 앞 실행의 것을 지우거나 덮어쓰지 않습니다. 한 종류가 실패해도 그 종류의 결과 zip 에 오류를 남기고 다음 종류로 넘어갑니다.

## 결과

종류마다 단계가 끝날 때마다 `내 드라이브/laya/result-<종류>-<시각>.zip` 을 새로 저장합니다(준비 단계는 `result-setup-<시각>.zip`). 중간에 실패해도 그때까지의 로그와 오류가 들어 있습니다. 이 zip 들을 PC 의 `Downloads/` 에 내려받아 Claude 에게 넘기면 됩니다. 담기는 것:

- `log.txt` — 모든 셀의 출력과 오류
- `results.json` — 환경, 데이터, 실행마다 설정, 모델마다 정확도·평균 확신도·ECE·로그 손실, 선택된 모델, 단계별 소요 시간
- `predictions.jsonl` — validation 질문마다 정답과 각 모델의 답·확신도 (상태 본문은 없음)
- `<실행>-laya-run.json`, `<실행>-train_report.json`, `<실행>-calibration.json` — 학습 설정, Laya 의 학습 기록, 맞춘 온도

선택된 체크포인트만 크기가 커서(종류마다 약 1.6 GB) `내 드라이브/laya/<종류>-<split>-<실행>-<시각>.zip` 에 따로 저장합니다. 9종이면 Drive 에 약 15 GB 가 필요합니다. 공간이 모자라면 그 종류의 체크포인트 저장만 실패로 남고 학습은 다음 종류로 이어집니다. Claude 에게는 결과 zip 만 넘기면 됩니다.

개선 루프(`laya/loop.py`)는 유료 라벨링을 하므로 여기가 아니라 PC 에서 돕니다.

In [ ]:
import contextlib
import gc
import json
import logging
import shutil
import sys
import time
import traceback
import zipfile
from pathlib import Path

from google.colab import drive

DRIVE = Path("/content/drive/MyDrive/laya")
STAMP = time.strftime("%Y%m%d-%H%M%S")
ENV = {}


def begin(kind):
    """한 종류(또는 준비 단계)의 결과 폴더를 새로 연다. 이 폴더가 통째로 그 종류의 결과 zip 이 된다."""
    global KIND, RESULT, LOG, SUMMARY
    KIND = kind
    RESULT = Path(f"/content/result-{kind}-{STAMP}")
    RESULT.mkdir()   # 이미 있으면 멈춘다: 다른 실행의 결과와 섞이지 않게
    LOG = RESULT / "log.txt"
    SUMMARY = {"kind": kind, "started": time.strftime("%Y%m%d-%H%M%S"), "seconds": {}, "env": dict(ENV)}


class Tee:
    """화면에 쓰는 것을 로그 파일에도 쓴다. 나머지 속성(isatty 등)은 화면 쪽 스트림의 것을 쓴다."""

    def __init__(self, screen, file):
        self.screen, self.file = screen, file

    def write(self, text):
        self.file.write(text)
        return self.screen.write(text)

    def flush(self):
        self.file.flush()
        self.screen.flush()

    def __getattr__(self, name):
        return getattr(self.screen, name)


def save():
    """지금까지의 결과를 Drive 의 결과 zip 으로 다시 쓴다(같은 실행이면 같은 이름)."""
    (RESULT / "results.json").write_text(json.dumps(SUMMARY, indent=1, default=str), encoding="utf-8")
    shutil.make_archive(str(DRIVE / RESULT.name), "zip", RESULT)


@contextlib.contextmanager
def logged(step):
    """한 단계: 출력과 오류를 log.txt 에 남기고, 성공하든 실패하든 끝에 결과 zip 을 저장한다."""
    started = time.time()
    with open(LOG, "a", encoding="utf-8") as f:
        handler = logging.StreamHandler(f)   # Laya 가 logging 으로 남기는 것도 받는다
        logging.getLogger().addHandler(handler)
        f.write(f"\n===== {step} ({time.strftime('%H:%M:%S')}) =====\n")
        try:
            with contextlib.redirect_stdout(Tee(sys.stdout, f)), contextlib.redirect_stderr(Tee(sys.stderr, f)):
                try:
                    yield
                except BaseException:
                    traceback.print_exc()
                    SUMMARY["failed"] = step
                    raise
        finally:
            logging.getLogger().removeHandler(handler)
            SUMMARY["seconds"][step] = round(time.time() - started, 1)
            f.flush()   # zip 에 이 단계의 로그(실패라면 traceback)까지 들어가도록
            save()


drive.mount("/content/drive")
if not (DRIVE / "colab.zip").exists():
    raise FileNotFoundError(f"{DRIVE}/colab.zip 이 없습니다. PC 에서 만든 zip 을 내 드라이브/laya/ 에 올려 주세요.")
%pip install -q laya==0.4.6 transformers==5.19.0

begin("setup")
with logged("setup"):
    zipfile.ZipFile(DRIVE / "colab.zip").extractall("/content")
    import importlib.metadata as md

    import torch

    PACKED = json.loads(Path("/content/laya/packed.json").read_text(encoding="utf-8"))
    KINDS = PACKED["kinds"]
    ENV.update({"packed": PACKED,
                "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
                if torch.cuda.is_available() else None,
                "python": sys.version.split()[0],
                **{p: md.version(p) for p in ("laya", "transformers", "torch", "huggingface_hub")}})
    SUMMARY["env"] = dict(ENV)
    print(json.dumps(ENV, indent=1))

In [ ]:
import tempfile

with logged("data"):
    # 압축을 푼 laya/ 의 모듈을 이름으로 불러온다. `import laya` 는 설치된 패키지 그대로다.
    if "/content/laya" not in sys.path:
        sys.path.insert(0, "/content/laya")
    # 같은 런타임에서 앞 실행이 불러 둔 laya/ 모듈은 버린다. 남아 있으면 새로 푼 코드 대신 옛 코드가 돈다.
    for source in Path("/content/laya").glob("*.py"):
        sys.modules.pop(source.stem, None)
    import joint
    import label
    import support
    import train

    # 이 실행의 체크포인트는 이 실행만의 폴더에 둔다. 같은 런타임에서 다시 돌려도 앞 실행의 폴더와
    # 이름이 겹치지 않고, 앞 실행이 Drive 에 저장하지 못한 체크포인트도 지우지 않는다.
    Path("/content/laya/checkpoints").mkdir(parents=True, exist_ok=True)
    train.CHECKPOINTS = Path(tempfile.mkdtemp(prefix=f"{STAMP}-", dir="/content/laya/checkpoints"))
    ENV["checkpoints"] = str(train.CHECKPOINTS)
    print("종류:", KINDS)


def load_data():
    """이 종류의 split 마다 상태·질문 수를 요약에 넣고, 채점에 쓸 validation 을 읽는다."""
    global VALIDATION
    SUMMARY["data"] = {}
    for split in train.SPLITS:
        path = label.DATA / KIND / f"{split}.jsonl"
        if path.exists():
            rows = label.read(path)
            SUMMARY["data"][split] = {"states": len(rows), "questions": sum(len(r["expected"]) for r in rows)}
    SUMMARY["max_len"] = train.MAX_LEN[KIND]
    VALIDATION = label.read(label.DATA / KIND / "validation.jsonl")
    print(KIND, SUMMARY["data"])

In [ ]:
# `train.py pack --picks <zip>[,<zip>...]` 로 만든 zip 이면, 학습 전에 Drive 의 support 체크포인트마다 coverage 의
# train·calibration 질문을 읽는다. 그 실행이 '일부'로 읽은 문단들이 joint 가 물을 보기다(joint.picks).
# 결과는 result-picks-<시각>.zip 의 coverage-predictions-<split>-<체크포인트 이름>.jsonl 이다.
# 한 체크포인트가 실패해도(Drive 에 없는 등) 그 오류를 남기고 다음 체크포인트로 넘어간다.
PICKS = PACKED.get("picks")
if PICKS:
    begin("picks")
    SUMMARY["picks"] = {}
    for name in PICKS["checkpoints"]:
        tag = Path(name).stem
        with contextlib.suppress(Exception), logged(f"picks {tag}"):
            picks_checkpoint = Path("/content/picks-checkpoint")
            shutil.rmtree(picks_checkpoint, ignore_errors=True)
            shutil.unpack_archive(str(DRIVE / name), str(picks_checkpoint), "zip")
            picks_agent = train.loaded(picks_checkpoint)
            SUMMARY["picks"][tag] = {}
            for split in PICKS["splits"]:
                SUMMARY["picks"][tag][split], records = support.evaluate(picks_checkpoint, picks_agent, split=split)
                with open(RESULT / f"coverage-predictions-{split}-{tag}.jsonl", "w", encoding="utf-8") as f:
                    for r in records:
                        f.write(json.dumps(r) + "\n")
                print("picks", tag, split, SUMMARY["picks"][tag][split])
        picks_agent = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 채점

종류마다 공개 체크포인트 두 개를 validation 에서 채점합니다. 정확도가 높은 쪽이 파인튜닝의 출발점입니다(계획 4단계). 길이 상한(`train.MAX_LEN`)을 넘는 질문은 자르지 않고 채점에서 뺍니다. 이 셀은 채점 함수만 정의하고, 실제 실행은 아래 파인튜닝 셀이 종류마다 합니다.

In [ ]:
import math


def ece(correct, confidence, bins=10):
    """확신도 구간마다 |정확도 - 평균 확신도| 를 질문 수로 가중 평균한 것."""
    total = 0.0
    for b in range(bins):
        inside = [i for i, c in enumerate(confidence) if b / bins < c <= (b + 1) / bins or (b == 0 and c == 0)]
        if inside:
            hit = sum(correct[i] for i in inside) / len(inside)
            total += len(inside) * abs(hit - sum(confidence[i] for i in inside) / len(inside))
    return total / len(confidence)


def log_loss(chances):
    """정답에 준 확률의 평균 -log(`train.chance`). 3지선다(support)에서 틀린 답의 1 - 확신도는 정답 확률이 아니므로 이것을 쓴다."""
    eps = 1e-6
    return sum(-math.log(min(max(p, eps), 1 - eps)) for p in chances) / len(chances)


def record(name, checkpoint):
    """한 모델을 채점해 정확도·확신도·ECE·로그 손실을 요약에 넣고, 질문별 답을 predictions.jsonl 로 다시 쓴다.
    ANSWERS, SURE 는 종류마다 새로 비운다(모델 이름 -> {(state_hash, 질문): 답 또는 확신도})."""
    sure, truth = {}, {}
    wrong, predicted = train.score(KIND, checkpoint, sure, truth)
    ANSWERS[name], SURE[name] = predicted, sure
    keys = list(predicted)
    correct = [k not in wrong for k in keys]
    confidence = [sure[k] for k in keys]
    SUMMARY.setdefault("accuracy", {})[name] = {
        "accuracy": round(sum(correct) / len(keys), 4), "wrong": len(wrong), "scored": len(keys),
        "mean_confidence": round(sum(confidence) / len(keys), 4), "ece": round(ece(correct, confidence), 4),
        "log_loss": round(log_loss([truth[k] for k in keys]), 4),
        "mean_confidence_when_wrong": round(sum(c for c, ok in zip(confidence, correct) if not ok)
                                            / max(1, len(wrong)), 4)}
    with open(RESULT / "predictions.jsonl", "w", encoding="utf-8") as f:
        for row in VALIDATION:
            for q, want in row["expected"].items():
                key = (row["provenance"]["state_hash"], q)
                f.write(json.dumps({"state_hash": key[0], "question": q, "expected": want,
                                    **{n: a.get(key) for n, a in ANSWERS.items()},
                                    **{f"{n}_confidence": s.get(key) for n, s in SURE.items()}}) + "\n")
    print(KIND, name, SUMMARY["accuracy"][name])

## 파인튜닝

fp16 과 gradient checkpointing 은 Laya 의 CUDA 기본값입니다. 유효 배치는 64 입니다: 보통 micro-batch 4 × 누적 16, 8,192 토큰인 `coverage` 는 1 × 64 (`train.MICRO_BATCH`). 옵션 순서와 무관하게 학습하도록 `option_layout="parallel"` 을 씁니다(`train.train` 의 기본).

종류마다 `RUNS` 의 실행마다 한 번씩 학습하고 validation 으로 고릅니다. 실행은 이름과 `TrainConfig` 값(`epochs`, `seed`, `label_smoothing`, `encoder_lr` 등)이고, `train` 을 주면 train 에서 그 수만큼만 무작위로(시드 0) 골라 씁니다(학습 곡선). Laya 0.4.6 에는 early stopping 이 없어서 에폭 수도 실행으로 나눕니다. calibration split 이 있으면 학습 뒤 그것으로 온도를 맞추고, 채점은 그 온도를 적용한 확신도로 합니다. 로그 손실이 가장 낮은 것이 선택됩니다. 이 모델은 확신할 때만 답하고 나머지는 미루므로(임계값 정책), 정확도만이 아니라 확신도가 맞고 틀림을 얼마나 가르는지도 봐야 합니다.

지금 `RUNS` 는 `useful` 실행 5 에서 앞선 두 설정입니다: label smoothing 0.1 로 2에폭과 3에폭(실행 5 의 선택). 1,000문항 남짓에서 실행 하나가 약 5–8분이고, `coverage` 는 더 깁니다. 선택되지 않은 체크포인트는 디스크를 비우려고 지웁니다.

In [ ]:
RUNS = [
    {"name": "e2-ls0.1", "epochs": 2, "label_smoothing": 0.1},
    {"name": "e3-ls0.1", "epochs": 3, "label_smoothing": 0.1},     # useful 실행 5 의 선택
]


def fit_kind():
    """이 종류를 제로샷 채점하고, RUNS 마다 학습·채점해 로그 손실이 가장 낮은 것을 Drive 에 저장한다."""
    global ANSWERS, SURE
    ANSWERS, SURE = {}, {}
    with logged("data"):
        load_data()
    with logged("zero-shot"):
        for name in train.PUBLISHED:
            record(name, name)
        base = max(train.PUBLISHED, key=lambda n: SUMMARY["accuracy"][n]["accuracy"])
        SUMMARY["base"] = base
        print(KIND, "출발점:", base)
    checkpoints = {}   # 모델 이름 -> 체크포인트 폴더
    SUMMARY["runs"] = {}
    for run in RUNS:
        config = dict(run)
        short, limit = config.pop("name"), config.pop("train", None)
        name = f"fine-tuned-{short}"
        with logged(f"fine-tune {short}"):
            split, rows = "train", SUMMARY["data"]["train"]["states"]
            if limit:
                # train 에서 limit 개 레코드를 시드 0 으로 골라 별도 split 으로 쓴다
                split, rows = train.subset(KIND, limit)
            done = train.train(KIND, [split], base=base, **config)
            # 다음 학습이 같은 폴더에 쓰지 않도록 실행 이름을 붙여 옮긴다
            checkpoints[name] = done.rename(done.with_name(f"{done.name}-{short}"))
            SUMMARY["runs"][name] = {**config, "train": rows}
            for file in ("laya-run.json", "train_report.json", "calibration.json"):
                if (checkpoints[name] / file).exists():
                    shutil.copy(checkpoints[name] / file, RESULT / f"{short}-{file}")
            if torch.cuda.is_available():
                SUMMARY["env"][f"peak_gpu_memory_gb_{short}"] = round(torch.cuda.max_memory_allocated() / 2**30, 1)
                torch.cuda.reset_peak_memory_stats()
        with logged(f"score {short}"):
            record(name, checkpoints[name])
    best = min(checkpoints, key=lambda n: SUMMARY["accuracy"][n]["log_loss"])
    SUMMARY["chosen"] = best
    print(KIND, "선택:", best, SUMMARY["accuracy"][best])
    if KIND in ("support", "joint"):
        # support 는 문단마다 답하고, 그 답을 합친 coverage 답을 coverage validation 에서 채점한다(support.decide).
        # joint 는 support 실행이 정하지 못한 질문에, 그 실행이 '일부'로 읽은 문단들을 함께 묻는다(joint/picks-validation.jsonl).
        with logged(f"coverage by {KIND}"):
            SUMMARY["coverage"], records = (support if KIND == "support" else joint).evaluate(checkpoints[best])
            with open(RESULT / "coverage-predictions.jsonl", "w", encoding="utf-8") as f:
                for r in records:
                    f.write(json.dumps(r) + "\n")
            print(f"coverage by {KIND}:", SUMMARY["coverage"])
    # 선택된 체크포인트(약 1.6 GB)만 결과 zip 과 따로 저장한다. 결과 zip 은 위 단계마다 이미 저장됐다.
    # 이름에 이 실행의 체크포인트 폴더 이름(시각과 고유 접미사)을 붙여 다른 실행의 zip 을 덮어쓰지 않는다.
    with logged("checkpoint"):
        SUMMARY["checkpoint_zip"] = shutil.make_archive(
            str(DRIVE / f"{checkpoints[best].name}-{train.CHECKPOINTS.name}"), "zip", checkpoints[best])
    for folder in checkpoints.values():
        shutil.rmtree(folder, ignore_errors=True)


DONE = {}   # 종류 -> 선택된 모델 또는 실패한 단계
for kind in KINDS:
    begin(kind)
    try:
        fit_kind()
        DONE[kind] = SUMMARY["chosen"]
    except Exception as error:
        # logged 는 단계 안의 오류를 log.txt 와 결과 zip 에 남기지만, 결과 zip 저장 자체가 실패하면 그 오류는
        # 어디에도 남지 않는다. 그래서 DONE 에 먼저 적고, 화면·로컬 log.txt·결과 zip 재저장에 차례로 남겨 본다.
        # 이 기록들은 각각 실패해도 다음 종류를 막지 않는다.
        SUMMARY.setdefault("failed", "saving results")
        DONE[kind] = f"failed at {SUMMARY['failed']}: {error!r} (로컬 기록 {RESULT})"
        with contextlib.suppress(Exception):
            traceback.print_exc()
        with contextlib.suppress(Exception), open(LOG, "a", encoding="utf-8") as f:
            f.write(f"\n===== 종류 실패 ({time.strftime('%H:%M:%S')}) =====\n")
            traceback.print_exc(file=f)
        with contextlib.suppress(Exception):
            save()
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    with contextlib.suppress(Exception):
        print(kind, "→", DONE[kind])
print(json.dumps(DONE, indent=1, ensure_ascii=False))
print("결과 zip:", sorted(str(p) for p in DRIVE.glob(f"result-*-{STAMP}.zip")))